# 填入运算符凑出 99 ― 让 Z3 去找括号的位置

## 本章要点

数的顺序是固定的，在中间填入 `+`, `-`, `*`，再用括号改变计算顺序。

- 反复做“把相邻两个数算出来合成一个”，任何括号方式都能表示
- 把每个阶段的数组设成 Z3 变量，用 `If` 选择运算
- 最后把找到的步骤还原成算式字符串

也能看出它与数字顺序自由的“凑出 24”问题的差别。

> 对下面的式子，在 `?` 处填入 `+`, `-`, `*` 中的任意一个，并合理放置括号，使式子的值等于 99。
>
> `1 ? 2 ? 3 ? 4 ? 5 ? 6 ? 7 ? 8 ? 9 = 99`

## 思路

不是从左往右一个个算，而是反复做**把相邻两个数算出来合成一个**的操作。

比如从 `1 2 3 4 5` 出发，

1. 计算 `3 + 4` → `1 2 7 5`
2. 计算 `1 * 2` → `2 7 5`
3. 计算 `7 + 5` → `2 12`
4. 计算 `2 - 12` → `-10`

这样每走一段，数的个数就减少一个。最后剩下的那个数就是式子的值。

```{mermaid}
graph
a1[1]
a2[2]
a3[3+4]
a4[5]
b1[1*2]
b2[3+4]
b3[5]
c1[1*2]
c2[3+4-5]
d1[1*2+3+4-5]
1 --> a1
2 --> a2
3 --> a3
4 --> a3
5 --> a4
a1 --> b1
a2 --> b1
a3 --> b2
a4 --> b3
b1 --> c1
b2 --> c2
b3 --> c2
c1 --> d1
c2 --> d1
```

让 Z3 去决定“把哪个位置、用哪个运算符合并”，就等于把所有括号方式都试了一遍。阶段共 $n-1$ 次，每个阶段可选的位置有“当时的元素个数 − 1”个。

## 一个阶段一个阶段地算

把阶段 `l` 的数组记作 `f[l]`。最初的 `f[0]` 是 `[1, 2, …, 9]`，每前进一个阶段就短一个。

`calc_tree(f, locs, ops)` 把每个阶段的计算摆成约束。在阶段 `l` 选择位置 `p = locs[l-1]` 时，

- `i < p` … `f[l][i] == f[l-1][i]`（原样保留）
- `i == p` … `f[l][p] == f[l-1][p] op f[l-1][p+1]`（在这里计算）
- `i > p` … `f[l][i] == f[l-1][i+1]`（从左边一位挪过来）

In [1]:
from z3 import *
from helper.z3 import all_solutions

OP_SYMBOLS = '+-*'   # 0:+, 1:-, 2:*


def calc_op(x1, x2, op):
    """按 op 的编号选择加、减、乘"""
    return If(op == 0, x1 + x2, If(op == 1, x1 - x2, x1 * x2))


def calc_tree(f, locs, ops):
    """把每个阶段的计算整理成约束列表返回"""
    exprs = []
    for l in range(1, len(f)):
        prev = f[l - 1]
        for i in range(len(f[l])):
            exprs.extend([
                Implies(i < locs[l - 1], f[l][i] == prev[i]),
                Implies(i == locs[l - 1], f[l][i] == calc_op(prev[i], prev[i + 1], ops[l - 1])),
                Implies(i > locs[l - 1], f[l][i] == prev[i + 1]),
            ])
    return exprs

### 用 If 选择运算符

```python
If(op == 0, x1 + x2, If(op == 1, x1 - x2, x1 * x2))
```

把 `If(条件, 为真时的值, 为假时的值)` 嵌套起来，切换 3 种运算。

| `op` | 计算 |
|---|---|
| 0 | `x1 + x2` |
| 1 | `x1 - x2` |
| 2 | `x1 * x2` |

条件一多嵌套就变深，选项多的时候用 `Function` 更好读（本章只有 3 个，用 `If` 足够了）。

## 显示成算式

Z3 给出的只是数，所以要准备一套还原算式的机制。`Expr` 是一个持有“左式、运算符、右式”的小类，`output()` 沿着各阶段的步骤拼出字符串。

当 `*` 或 `-` 的右侧出现 `+` 或 `-` 时要加括号，这样才能把 `1 + 2 * 3` 和 `(1 + 2) * 3` 区分开。

In [2]:
class Expr:
    """把计算树变成带括号的字符串"""
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

    def __str__(self):
        # 左侧：* 的左边是 + - 时要加括号
        if isinstance(self.left, Expr) and self.op == '*' and self.left.op in '+-':
            left = f'({self.left})'
        else:
            left = f'{self.left}'

        # 右侧：* 或 - 的右边是 + - 时要加括号
        if isinstance(self.right, Expr) and self.op in '*-' and self.right.op in '+-':
            right = f'({self.right})'
        else:
            right = f'{self.right}'

        return f'{left} {self.op} {right}'


def output(numbers, locs, ops):
    """从数的顺序和步骤生成算式字符串"""
    exprs = [str(x) for x in numbers]
    for loc, op in zip(locs, ops):
        exprs[loc] = Expr(exprs[loc], OP_SYMBOLS[op], exprs[loc + 1])
        del exprs[loc + 1]
    return str(exprs[0])

## 用小例子确认

给 `1 2 3 4 5` 套一个固定的步骤。

- 第 1 段：位置 2 用 `+`（`3 + 4`）
- 第 2 段：位置 0 用 `*`（`1 * 2`）
- 第 3 段：位置 1 用 `-`（`7 - 5`）
- 第 4 段：位置 0 用 `+`（`2 + 2`）

结果应该是 `1 * 2 + ((3 + 4) - 5) = 4`。

In [3]:
n = 5
f = [[Int(f'f{l}_{i}') for i in range(n - l)] for l in range(n)]

solver = Solver()
for i in range(n):
    solver.add(f[0][i] == i + 1)      # 一开始是 1, 2, 3, 4, 5

locs = [2, 0, 1, 0]                   # 合并哪个位置
ops = [0, 2, 1, 0]                    # +, *, -, +
solver.add(calc_tree(f, locs, ops))

assert solver.check() == sat
m = solver.model()

expr = output([i + 1 for i in range(n)], locs, ops)
print(expr, "=", eval(expr))

print("各阶段的值")
for level in range(n):
    print([m[f[level][i]] for i in range(n - level)])

1 * 2 + 3 + 4 - 5 = 4
各阶段的值
[1, 2, 3, 4, 5]
[1, 2, 7, 5]
[2, 7, 5]
[2, 2]
[4]


和手算一样，得到了 `1 * 2 + 3 + 4 - 5 = 4`（作为计算树是 `1 * 2 + ((3 + 4) - 5)`）。从各阶段的值也能看出数一个一个减少的过程。

## 寻找凑成 99 的组合

这次让 Z3 去决定位置和运算符。因为共有 9 个数，约束是“位置 `locs[i]` 不小于 0、小于 `9-i-1`”“运算符为 0〜2”。

In [4]:
n = 9
f = [[Int(f'f{l}_{i}') for i in range(n - l)] for l in range(n)]

solver = Solver()
for i in range(n):
    solver.add(f[0][i] == i + 1)          # 从 1 到 9

locs = IntVector('loc', n - 1)            # 各阶段合并的位置
ops = IntVector('op', n - 1)              # 各阶段的运算符

for i, loc in enumerate(locs):
    solver.add(0 <= loc, loc < n - i - 1)  # 到该阶段元素个数 - 1 为止

for op in ops:
    solver.add(0 <= op, op <= 2)           # 0:+, 1:-, 2:*

solver.add(calc_tree(f, locs, ops))
solver.add(f[-1][0] == 99)                 # 最后剩下的数是 99

print(solver.check())

m = solver.model()
print("各阶段的值")
for level in range(n):
    print([m[f[level][i]] for i in range(n - level)])

locs_used = [m[x].as_long() for x in locs]
ops_used = [m[x].as_long() for x in ops]
print("位置:", locs_used)
print("运算符:", [OP_SYMBOLS[x] for x in ops_used])

sat
各阶段的值
[1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2, -1, 5, 6, 7, 8, 9]
[1, 2, -1, 5, 6, 7, 17]
[1, 2, -1, 5, 6, 119]
[1, 2, -1, 5, -113]
[1, 3, 5, -113]
[1, 15, -113]
[1, -98]
[99]
位置: [2, 6, 5, 4, 1, 1, 1, 0]
运算符: ['-', '+', '*', '-', '-', '*', '+', '-']


最后，把找到的步骤还原成算式。

In [5]:
expr = output(list(range(1, 10)), locs_used, ops_used)
print(expr, "=", eval(expr))

1 - ((2 - (3 - 4)) * 5 + 6 - 7 * (8 + 9)) = 99


比如会得到下面这样的式子（每次运行可能得到不同的式子）。

```
1 - ((2 - (3 - 4)) * 5 + 6 - 7 * (8 + 9)) = 99
```

Z3 找到的是**满足条件的第一个**，所以式子形状每次不一定相同。但不管哪个都等于 99。

## 小结

- 对固定的数列，括号的添加方式可以表示成“合并相邻两个”的操作序列
- 每个阶段准备一个数组，用 `calc_tree()` 把“保留、计算、移位”三种情况写成 `Implies`
- 运算符编成整数编号，用 `If` 切换计算
- 找到的步骤还原成计算树（`Expr`）就能读成算式
- 数的顺序也自由的“凑出 24”问题，还要多一条排列约束（上一章）